# GraphSAGE — Edge Classification Benchmark
## ToN-IoT · 10-Class Edge-Level Classification



# Section 0 — Configuration

In [1]:
GRAPH_PT_PATH = "/content/drive/MyDrive/toniot_edge_graph.pt"
SEED = 42
MODEL_NAME = "GraphSAGE"

# ── GraphSAGE Hyperparameters ────────────────────────────────────────────────
HP = dict(
    hidden_dim      = 64,
    num_layers      = 3,
    dropout         = 0.3,
    lr              = 5e-4,
    weight_decay    = 1e-4,
    batch_size      = 2048,
    fan_out         = [10, 10, 10],
    max_epochs      = 100,
    patience        = 15,
    min_delta       = 5e-5,
    val_ratio       = 0.10,
    label_smoothing = 0.03,
)
print('Hyperparameters:', HP)


Hyperparameters: {'hidden_dim': 64, 'num_layers': 3, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 2048, 'fan_out': [10, 10, 10], 'max_epochs': 100, 'patience': 15, 'min_delta': 5e-05, 'val_ratio': 0.1, 'label_smoothing': 0.03}


# Section 1 — Install Dependencies

In [2]:
import sys, subprocess, re

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "-q", *args])

subprocess.call([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "torch-geometric", "pyg-lib", "torch-scatter",
    "torch-sparse", "torch-cluster", "torch-spline-conv"
])

import torch

torch_ver = re.match(r"(\d+\.\d+\.\d+)", torch.__version__).group(1)
cuda_ver  = torch.version.cuda

if cuda_ver is None:
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+cpu.html"
else:
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver.replace('.','')}.html"

pip_install(
    "torch-geometric", "pyg-lib", "torch-scatter",
    "torch-sparse", "torch-cluster", "torch-spline-conv",
    "-f", wheel_url
)
print("Installed PyG from:", wheel_url)


Installed PyG from: https://data.pyg.org/whl/torch-2.11.0+cu128.html


# Section 2 — Imports & Reproducibility

In [3]:
from torch_geometric.nn import SAGEConv

import os, time, warnings
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import LinkNeighborLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')

def set_seed(seed: int = 42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")


Device: cuda


# Section 3 — Load Graph & Normalise (train-split only)

In [6]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

data = torch.load(GRAPH_PT_PATH, map_location='cpu', weights_only=False)
print(data)

print(f"\nNodes         : {data.num_nodes:,}")
print(f"Edges (total) : {data.edge_index.shape[1]:,}")
print(f"Node feat dim : {data.x.shape[1]}")
print(f"Edge feat dim : {data.edge_attr.shape[1]}")

NUM_CLASSES   = int(data.edge_y.max().item()) + 1
NODE_FEAT_DIM = int(data.x.shape[1])
EDGE_FEAT_DIM = int(data.edge_attr.shape[1])

CLASS_NAMES = [
    "backdoor","ddos","dos","injection","mitm",
    "normal","password","ransomware","scanning","xss"
]

# Normalise node features on TRAIN split only (no leakage)
train_node_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
train_src = data.edge_index[0][data.edge_train_mask]
train_dst = data.edge_index[1][data.edge_train_mask]
train_node_mask[train_src] = True
train_node_mask[train_dst] = True

x_train = data.x[train_node_mask]
mean = x_train.mean(dim=0)
std  = x_train.std(dim=0) + 1e-8
data.x = (data.x - mean) / std

# Normalise edge features on TRAIN edges only (no leakage)
ea = data.edge_attr.clone()
for col in [0, 1, 2, 5, 6, 7]:
    m = ea[data.edge_train_mask, col].mean()
    s = ea[data.edge_train_mask, col].std() + 1e-8
    ea[:, col] = (ea[:, col] - m) / s
data.edge_attr = ea

train_edge_idx = data.edge_train_mask.nonzero(as_tuple=False).view(-1).long()
val_edge_idx   = data.edge_val_mask.nonzero(as_tuple=False).view(-1).long()
test_edge_idx  = data.edge_test_mask.nonzero(as_tuple=False).view(-1).long()

def describe_split(name, edge_idx):
    labels = data.edge_y[edge_idx]
    counts = torch.bincount(labels, minlength=NUM_CLASSES)
    total  = int(edge_idx.numel())
    txt    = ", ".join([f"{CLASS_NAMES[i]}={int(counts[i])}" for i in range(NUM_CLASSES)])
    print(f"{name:<6} edges: {total:>10,} | {txt}")

describe_split("Train", train_edge_idx)
describe_split("Val",   val_edge_idx)
describe_split("Test",  test_edge_idx)

# Fallback (src, dst) -> edge-id lookup for edge-attr resolver
edge_pair_to_id = {}
for eid, (s, d) in enumerate(data.edge_index.t().tolist()):
    edge_pair_to_id.setdefault((int(s), int(d)), int(eid))
print(f"Built edge pair lookup for {len(edge_pair_to_id):,} unique directed pairs.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data(x=[2262, 8], edge_index=[2, 181052], edge_attr=[181052, 8], y=[2262], edge_y=[181052], edge_train_mask=[181052], edge_val_mask=[181052], edge_test_mask=[181052])

Nodes         : 2,262
Edges (total) : 181,052
Node feat dim : 8
Edge feat dim : 8
Train  edges:    126,736 | backdoor=14000, ddos=14000, dos=14000, injection=14000, mitm=736, normal=14000, password=14000, ransomware=14000, scanning=14000, xss=14000
Val    edges:     27,157 | backdoor=3000, ddos=3000, dos=3000, injection=3000, mitm=157, normal=3000, password=3000, ransomware=3000, scanning=3000, xss=3000
Test   edges:     27,159 | backdoor=3000, ddos=3000, dos=3000, injection=3000, mitm=159, normal=3000, password=3000, ransomware=3000, scanning=3000, xss=3000
Built edge pair lookup for 3,013 unique directed pairs.


# Section 4 — Loss (unweighted)


In [7]:
# Unweighted loss — no inverse-frequency correction
# (class_weights removed to study the impact on minority-class recall)

criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])
print(f"Loss: CrossEntropyLoss(label_smoothing={HP['label_smoothing']})")


Loss: CrossEntropyLoss(label_smoothing=0.03)


# Section 5 — DataLoaders (structural leakage fix)

In [8]:
# ── Structural-leakage fix (matches BoT-IoT benchmark) ──────────────────────
# Train/val loaders use a train-only MP graph so test edges never appear
# in neighbourhood sampling during training.
# Test loader uses the full graph (training complete, no gradient flow).

def make_loader(edge_idx, batch_size, shuffle, mp_edge_idx=None):
    edge_idx = edge_idx.long()
    if mp_edge_idx is not None:
        mp_edge_index = data.edge_index[:, mp_edge_idx.long()]
    else:
        mp_edge_index = data.edge_index

    from torch_geometric.data import Data as PygData
    mp_data = PygData(
        x=data.x,
        edge_index=mp_edge_index,
        edge_attr=data.edge_attr,
        num_nodes=data.num_nodes,
    )
    return LinkNeighborLoader(
        mp_data,
        num_neighbors=HP['fan_out'],
        edge_label_index=data.edge_index[:, edge_idx],
        edge_label=data.edge_y[edge_idx].long(),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
    )

train_loader = make_loader(train_edge_idx, HP['batch_size'],     shuffle=True,  mp_edge_idx=train_edge_idx)
val_loader   = make_loader(val_edge_idx,   HP['batch_size'] * 2, shuffle=False, mp_edge_idx=train_edge_idx)
test_loader  = make_loader(test_edge_idx,  HP['batch_size'] * 2, shuffle=False, mp_edge_idx=None)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")

def resolve_edge_attr(batch, seed_edge_idx, full_edge_attr, pair_lookup):
    if hasattr(batch, 'input_id') and batch.input_id is not None:
        global_ids = seed_edge_idx.cpu()[batch.input_id.cpu().long()]
        return full_edge_attr[global_ids].to(batch.x.device)
    edge_label_index = batch.edge_label_index
    if hasattr(batch, 'n_id') and int(edge_label_index.max()) < int(batch.n_id.numel()):
        src = batch.n_id[edge_label_index[0].cpu().long()]
        dst = batch.n_id[edge_label_index[1].cpu().long()]
    else:
        src = edge_label_index[0].cpu().long()
        dst = edge_label_index[1].cpu().long()
    edge_ids = torch.tensor(
        [pair_lookup[(int(s), int(d))] for s, d in zip(src.tolist(), dst.tolist())],
        dtype=torch.long
    )
    return full_edge_attr[edge_ids].to(batch.x.device)

print("Edge-attribute resolver ready.")


Train batches: 62
Val   batches: 7
Test  batches: 7
Edge-attribute resolver ready.


## Model Definition — Ablation Model (only model kept)

In [9]:
from torch_geometric.nn import GCNConv

class GCNStyleAblation(nn.Module):
    """
    Ablation of GraphSAGEEdgeClassifier: identical pipeline, except the
    message-passing operator is GCNConv (self-loop folded into the same
    symmetric-normalised average as neighbours) instead of SAGEConv
    (explicit concatenated self-pathway, Eq. 3).
    Encoder, edge MLP, depth, width, dropout are all unchanged.
    """
    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim,
                 num_layers, num_classes, dropout):
        super().__init__()
        self.dropout      = dropout
        self.node_encoder = nn.Linear(node_feat_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))   # <- only change vs. GraphSAGEEdgeClassifier
            self.norms.append(nn.LayerNorm(hidden_dim))
        mlp_in = hidden_dim * 2 + edge_feat_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim * 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def encode_nodes(self, x, edge_index):
        h = F.relu(self.node_encoder(x))
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index); h = norm(h); h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

    def forward(self, x, edge_index, edge_attr, edge_label_index):
        h = self.encode_nodes(x, edge_index)
        src, dst = edge_label_index
        return self.edge_mlp(torch.cat([h[src], h[dst], edge_attr], dim=-1))


## Train / Eval Functions

In [10]:
def train_epoch():
    model.train()
    total_loss, total_acc, n = 0., 0., 0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        ea    = resolve_edge_attr(batch, train_edge_idx, data.edge_attr, edge_pair_to_id)
        logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
        y      = batch.edge_label
        loss   = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bs          = y.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == y).float().sum().item()
        n          += bs
    return total_loss / n, total_acc / n

@torch.no_grad()
def evaluate_split(loader):
    model.eval()
    # Pick the right edge_idx for the attr resolver
    if loader is train_loader:
        eidx = train_edge_idx
    elif loader is val_loader:
        eidx = val_edge_idx
    else:
        eidx = test_edge_idx
    total_loss, total_acc, n = 0., 0., 0
    all_preds, all_labels = [], []
    for batch in loader:
        batch  = batch.to(DEVICE)
        ea     = resolve_edge_attr(batch, eidx, data.edge_attr, edge_pair_to_id)
        logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
        y      = batch.edge_label
        loss   = criterion(logits, y)
        bs          = y.size(0)
        total_loss += loss.item() * bs
        total_acc  += (logits.argmax(-1) == y).float().sum().item()
        n          += bs
        all_preds.append(logits.argmax(-1).cpu())
        all_labels.append(y.cpu())
    f1 = f1_score(torch.cat(all_labels).numpy(), torch.cat(all_preds).numpy(),
                  average='macro', zero_division=0)
    return total_acc / n, total_loss / n, f1


## Multi-Seed Evaluation — GraphSAGE_GCNStyleAblation Ablation (ton_iot, unweighted_loss)

In [11]:
# ============================================================
# Section 16 — Multi-Seed Evaluation (GraphSAGE, ton_iot, unweighted_loss)
# ============================================================
# NOTE: train_epoch()/evaluate_split() in this notebook close over the GLOBAL
# `model`, `optimizer`, `criterion`, `train_loader`, `val_loader`, `test_loader`.
# This cell reassigns those globals each seed iteration so the original
# functions can be reused unmodified.
import copy, json, os
import numpy as np
import pandas as pd

MODEL_CLASS  = GCNStyleAblation
MODEL_TAG    = "GraphSAGE_GCNStyleAblation"
DATASET_TAG  = "ton_iot"
STRATEGY_TAG = "unweighted_loss"

SEEDS = [0, 1]

def build_criterion(train_edge_idx_r):
    train_labels_r = data.edge_y[train_edge_idx_r].long()
    counts_r = torch.bincount(train_labels_r, minlength=NUM_CLASSES).float().clamp(min=1)
    return nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])

def run_once(seed):
    global model, optimizer, scheduler, criterion, train_loader, val_loader, test_loader
    set_seed(seed)

    train_edge_idx_r = train_edge_idx
    val_edge_idx_r   = val_edge_idx

    criterion = build_criterion(train_edge_idx_r)

    train_loader = make_loader(train_edge_idx_r, HP['batch_size'],     shuffle=True,  mp_edge_idx=train_edge_idx_r)
    val_loader   = make_loader(val_edge_idx_r,   HP['batch_size'] * 2, shuffle=False, mp_edge_idx=train_edge_idx_r)
    test_loader  = make_loader(test_edge_idx,    HP['batch_size'] * 2, shuffle=False, mp_edge_idx=None)

    model = MODEL_CLASS(
        node_feat_dim = NODE_FEAT_DIM,
        edge_feat_dim = EDGE_FEAT_DIM,
        hidden_dim    = HP['hidden_dim'],
        num_layers    = HP['num_layers'],
        num_classes   = NUM_CLASSES,
        dropout       = HP['dropout'],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=HP['lr'], weight_decay=HP['weight_decay'])
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

    best_val_f1, patience_count, best_state, best_epoch = -float('inf'), 0, None, 0
    for epoch in range(1, HP['max_epochs'] + 1):
        tr_loss, tr_acc = train_epoch()
        val_acc, val_loss, val_f1 = evaluate_split(val_loader)
        scheduler.step(val_f1)

        if val_f1 > best_val_f1 + HP['min_delta']:
            best_val_f1, best_epoch, patience_count = val_f1, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
        if patience_count >= HP['patience']:
            break

    model.load_state_dict(best_state)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            ea = resolve_edge_attr(batch, test_edge_idx, data.edge_attr, edge_pair_to_id)
            logits = model(batch.x, batch.edge_index, ea, batch.edge_label_index)
            all_preds.append(logits.argmax(-1).cpu())
            all_labels.append(batch.edge_label.cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    test_acc       = accuracy_score(y_true, y_pred)
    test_macro_f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    test_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    test_recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)

    per_class_f1        = f1_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_recall    = recall_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))

    per_class = {
        CLASS_NAMES[i]: dict(f1=float(per_class_f1[i]), precision=float(per_class_precision[i]), recall=float(per_class_recall[i]))
        for i in range(NUM_CLASSES)
    }

    return dict(
        seed=seed, epochs_ran=best_epoch,
        acc=float(test_acc), macro_f1=float(test_macro_f1),
        precision=float(test_precision), recall=float(test_recall),
        per_class=per_class,
    )

results = [run_once(s) for s in SEEDS]

def agg_mean_std(vals):
    arr = np.array(vals, dtype=float)
    return dict(mean=float(arr.mean()), std=float(arr.std(ddof=1)) if len(arr) > 1 else 0.0)

overall = {metric: agg_mean_std([r[metric] for r in results]) for metric in ['acc', 'macro_f1', 'precision', 'recall']}

per_class_agg = {}
for cls in CLASS_NAMES:
    per_class_agg[cls] = {metric: agg_mean_std([r['per_class'][cls][metric] for r in results]) for metric in ['f1', 'precision', 'recall']}

summary = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS, n_seeds=len(SEEDS),
    per_seed_results=results, overall=overall, per_class=per_class_agg,
)

print(f"\n{'='*60}")
print(f"  {MODEL_TAG} — {DATASET_TAG} / {STRATEGY_TAG} — {len(SEEDS)}-seed summary")
print(f"{'='*60}")
for metric, stat in overall.items():
    print(f"  {metric:<10}: {stat['mean']:.4f} ± {stat['std']:.4f}")
print("\n  Per-class F1 (mean ± std):")
for cls, stat in per_class_agg.items():
    f1s = stat['f1']
    print(f"    {cls:<16}: {f1s['mean']:.4f} ± {f1s['std']:.4f}")

out_dir = "/content/results" if os.path.isdir("/content") else "."
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/{DATASET_TAG}_{STRATEGY_TAG}_{MODEL_TAG}_multiseed.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved: {out_path}")

try:
    from google.colab import files as colab_files
    colab_files.download(out_path)
except ImportError:
    pass



  GraphSAGE_GCNStyleAblation — ton_iot / unweighted_loss — 2-seed summary
  acc       : 0.9004 ± 0.0002
  macro_f1  : 0.8119 ± 0.0001
  precision : 0.8094 ± 0.0000
  recall    : 0.8152 ± 0.0002

  Per-class F1 (mean ± std):
    backdoor        : 0.9978 ± 0.0006
    ddos            : 0.8553 ± 0.0045
    dos             : 0.9729 ± 0.0000
    injection       : 0.7611 ± 0.0060
    mitm            : 0.0000 ± 0.0000
    normal          : 0.9921 ± 0.0032
    password        : 0.7119 ± 0.0139
    ransomware      : 0.9894 ± 0.0009
    scanning        : 0.9515 ± 0.0006
    xss             : 0.8871 ± 0.0010

Saved: /content/results/ton_iot_unweighted_loss_GraphSAGE_GCNStyleAblation_multiseed.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>